# 🏆 Dashboard de Inteligencia de Mercado — Lujo y Activaciones VIP, Mundial 2026

**Proyecto II · Módulo II — Análisis y Visualización de Datos**

**Cliente ficticio:** Consultora dedicada a posicionar marcas de lujo en eventos que le aporten el revenue adecuado.  Disponen de métricas de precios de mercado y de ventas, pero no resulta fácil con estas métricas estáticas hacer predicciones ni tomar decisiones de negocio como en qué evento promocionar y posicionar qué producto. Idea de aprovechar el Mundial  de Futbol para posicionar dos o tres de sus productos de diferentes segmentos en el nicho de mercado de aficionados al mundial que asisten a zonas VIP, invitados y celebrities de los estadios de USA. (México y Canadá se aplicarán los mismos criterios que en USA aunque sus datos fueran diferentes ya que se hará una única campaña).

**Pregunta de negocio:** ¿Qué marcas, modelos y rangos de precio de relojes de lujo representan la mejor oportunidad de posicionamiento de marca frente al público VIP del Mundial 2026, dado que ninguna marca de alta relojería es actualmente patrocinador oficial?

**Dataset elegido:** *Luxury Watch Listings* (más de 280.000 anuncios de venta de relojes de lujo, scraping de Chrono24)
🔗 https://www.kaggle.com/datasets/philmorekoung11/luxury-watch-listings

**Marcas incluidas:** Rolex, Omega, Patek Philippe, Audemars Piguet, Breitling, Tudor, Cartier, Panerai, IWC, Seiko, Jaeger-LeCoultre, TAG Heuer, Hublot, Zenith, Vacheron Constantin, Longines, A. Lange & Söhne, Richard Mille, Breguet, Ulysse Nardin, Hamilton, NOMOS, Oris, Sinn.

---

## 📅 Contenido de este cuaderno

| | Bloque | Contenido |
|---|---|---|
| | Carga y exploración | Dimensiones, tipos de columna, mapeo de variables |
| | Calidad del dato | Nulos, duplicados, limpieza justificada |
| | Variable de negocio | Creación del "Tier de Lujo" |
| | Estadística descriptiva | Medidas de tendencia central y dispersión por marca |
| | Gobernanza inicial | Mapeo frente a patrocinadores oficiales del Mundial 2026 |
| | Correlaciones | Matriz de correlación de variables numéricas |
| | Storytelling visual | 4 gráficos interactivos con Plotly |
| | Sesgos | Borrador de la sección de gobernanza y sesgos |

> 💡 **Cómo usar este cuaderno:** ejecuta las celdas en orden, de arriba a abajo. Las celdas marcadas con ⚙️ requieren que revises o ajustes un valor antes de continuar (por ejemplo, nombres de columnas, ya que pueden variar ligeramente según la versión del dataset descargado).


---
# 🟦 — Exploración y preparación del dato


## 0. Configuración del entorno

Importamos las librerías que usaremos durante todo el proyecto. `plotly` viene preinstalado en Google Colab; `statsmodels` se necesita para las líneas de tendencia de los gráficos


In [50]:
# Librerías base
import pandas as pd
import numpy as np
import re
import os

# Librerías de visualización
import plotly.express as px
import plotly.graph_objects as go

# Necesario para las líneas de tendencia (trendline) de Plotly
!pip install -q statsmodels

# Configuración de visualización en Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("Librerías cargadas correctamente ✅")

Librerías cargadas correctamente ✅


## 1. Descarga del dataset

El dataset **Luxury Watch Listings** está alojado en Kaggle:
🔗 https://www.kaggle.com/datasets/philmorekoung11/luxury-watch-listings

Whatches.csv lo colocamos en la carpeta "archivos".


In [51]:
# Definimos el nombre del archivo de datos de forma robusta
from pathlib import Path

rutas_candidatas = [
    Path("Watches.csv"),
    Path("Mundial_Luxury_Stands") / "Watches.csv",
    Path(r"C:\Users\EVO\Desktop\BOOTCAMP\MODULO II\Proyecto II\Mundial_Luxury_Stands\Watches.csv"),
]

ruta_csv = next((ruta for ruta in rutas_candidatas if ruta.exists()), None)
if ruta_csv is None:
    raise FileNotFoundError("No se ha encontrado Watches.csv en las rutas esperadas.")

# Cargamos el DataFrame
df = pd.read_csv(ruta_csv, low_memory=False)

print(f"Dataset cargado correctamente desde: {ruta_csv}")
print(f"Dataset cargado correctamente: {df.shape[0]:,} filas x {df.shape[1]} columnas")


Dataset cargado correctamente desde: Watches.csv
Dataset cargado correctamente: 284,491 filas x 14 columnas


## 2. Limpieza y conversión de variables
Revisado el dataset limpiamos y transformamos la columna **Yop** y la columna **price** ya que no son numéricas porque contiene algunos datos como texto, y nos interesa el Yop (year of production) para elegir los modelos de relojes más recientes del mercado y el precio para el posicionamiento en el segmento que queremos.

In [52]:
# ============================================================
#  LIMPIEZA Y CONVERSIÓN DE VARIABLES
# ============================================================

# 1. Transformación de 'yop' (Año de producción)
# El CSV contiene valores como "2023 (Approximation)" y "Unknown".
# pd.to_numeric(..., errors='coerce') convertía todos los años aproximados
# en nulos; extraemos primero el año para aprovechar esos registros.
def extraer_yop(valor):
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip()
    match = re.search(r'((?:19|20)\d{2})', texto)
    if not match:
        return np.nan
    anio = int(match.group(1))
    return anio if 1900 <= anio <= 2026 else np.nan


df['yop_raw'] = df['yop']
df['yop'] = df['yop_raw'].apply(extraer_yop).astype('Int64')
df['yop_fuente'] = np.select(
    [
        df['yop'].notna() & df['yop_raw'].astype('string').str.contains('Approximation', case=False, na=False),
        df['yop'].notna(),
    ],
    ['aproximado_extraido', 'original_exacta'],
    default='sin_dato'
)


# 2. Función para limpiar precios
def limpiar_precio(valor):
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)

    texto = str(valor)
    # Nos quedamos solo con dígitos, comas y puntos
    texto_limpio = re.sub(r'[^\d.,]', '', texto)
    # Asumimos formato anglosajón: la coma es separador de miles
    texto_limpio = texto_limpio.replace(',', '')

    try:
        return float(texto_limpio) if texto_limpio else np.nan
    except ValueError:
        return np.nan


# 3. Aplicamos la función directamente sobre 'price'
columna_precio_origen = 'price_raw' if 'price_raw' in df.columns else 'price'
df['price'] = df[columna_precio_origen].apply(limpiar_precio).astype('Float64')


# 4. Reporte de resultados para control
print("¡Transformaciones completadas con éxito! 🚀")
print(f" -> Tipo de 'yop': {df['yop'].dtype}")
print(f" -> Tipo de 'price': {df['price'].dtype}\n")
print(f"Años de producción recuperados: {df['yop'].notna().sum():,} de {len(df):,} filas")
print(df['yop_fuente'].value_counts())
print(f"Precios convertidos correctamente: {df['price'].notna().sum():,} de {len(df):,} filas")

print("\nEstadísticos iniciales de precio (sin limpiar outliers todavía):")
print(df['price'].describe())


¡Transformaciones completadas con éxito! 🚀
 -> Tipo de 'yop': Int64
 -> Tipo de 'price': Float64

Años de producción recuperados: 188,290 de 284,491 filas
yop_fuente
original_exacta        148058
sin_dato                96201
aproximado_extraido     40232
Name: count, dtype: int64
Precios convertidos correctamente: 269,826 de 284,491 filas

Estadísticos iniciales de precio (sin limpiar outliers todavía):
count     269,826.00
mean       18,509.62
std        65,617.86
min            10.00
25%         3,021.00
50%         6,899.00
75%        15,902.75
max     9,000,000.00
Name: price, dtype: Float64


## 3. Exploración inicial (Checklist I)

### 3.1. Documentamos de forma técnica las dimensiones, los tipos de columna y el volumen de registros, tal y como exige la primera parte del checklist de evaluación.


In [53]:
print("="*60)
print("FICHA TÉCNICA DEL DATASET")
print("="*60)
print(f"Número de registros (filas):   {df.shape[0]:,}")
print(f"Número de variables (columnas): {df.shape[1]}")
print(f"Memoria utilizada:              {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

print("\nColumnas y tipos de dato:")
print(df.dtypes)

print("\nPrimeras filas:")
df.head()

FICHA TÉCNICA DEL DATASET
Número de registros (filas):   284,491
Número de variables (columnas): 16
Memoria utilizada:              208.20 MB

Columnas y tipos de dato:
Unnamed: 0      int64
name           object
price         Float64
brand          object
model          object
ref            object
mvmt           object
casem          object
bracem         object
yop             Int64
cond           object
sex            object
size           object
condition      object
yop_raw        object
yop_fuente     object
dtype: object

Primeras filas:


,Unnamed: 0,name,price,brand,model,ref,mvmt,casem,bracem,yop,cond,sex,size,condition,yop_raw,yop_fuente
0,0,Audemars Piguet Royal Oak Offshore Chronograph...,"43,500.00",Audemars Piguet,Royal Oak Offshore Chronograph,26237ST.OO.1000ST.01,NaN,NaN,NaN,2019,Unworn,Men's watch/Unisex,42 mm,NaN,2019,original_exacta
1,1,Audemars Piguet Royal Oak Selfwinding\n39mm Bl...,"71,500.00",Audemars Piguet,Royal Oak Selfwinding,15300ST.OO.1220ST.02,NaN,NaN,NaN,2012,Very good,Men's watch/Unisex,39 mm,NaN,2012,original_exacta
2,2,Audemars Piguet Royal Oak Chronograph\nBlue Di...,"79,191.00",Audemars Piguet,Royal Oak Chronograph,26331ST,Automatic,Steel,Steel,<NA>,Unworn,NaN,41 mm,NaN,Unknown,sin_dato
3,3,Audemars Piguet Royal Oak Chronograph\nSelfwin...,"108,000.00",Audemars Piguet,Royal Oak Chronograph,26715ST.OO.1356ST.01,Automatic,Steel,Steel,2022,New,Men's watch/Unisex,38 mm,NaN,2022 (Approximation),aproximado_extraido
4,4,Audemars Piguet Royal Oak Offshore Chronograph...,"27,500.00",Audemars Piguet,Royal Oak Offshore Chronograph,26170ST.OO.1000ST.01,Automatic,Steel,Steel,<NA>,Very good,Men's watch/Unisex,42 x 54 mm,NaN,Unknown,sin_dato


### 3.2. Variable "Condition"
Dado que el dataset proviene de un scrapping de una página de venta de relojes de lujo, pero se vendian relojes de segunda mano aparte de nuevos, para los objetivos del proyecto solo nos ineresan los anuncios de relojes nuevos sobre los que calcular, precios por marca, diferentes segmentos de lujo, etc.

In [54]:
# ============================================================
# ANÁLISIS DE LA VARIABLE CONDICIÓN
# ============================================================
# El CSV de Chrono24 puede tener la condición en una columna
# llamada 'condition' O 'cond'. Detectamos cuál existe y tiene datos.

# 1. Detección robusta del nombre real de la columna
col_cond = None
for candidato in ['condition', 'cond']:
    if candidato in df.columns and df[candidato].notna().any():
        col_cond = candidato
        print(f"✅ Columna de condición detectada: '{col_cond}'")
        break

if col_cond is None:
    print("⚠️  No se encontró columna de condición con datos. Se usarán todos los registros.")
else:
    total_opciones = df[col_cond].nunique()
    conteo_condicion  = df[col_cond].value_counts(dropna=False)
    porcentaje_condicion = df[col_cond].value_counts(dropna=False, normalize=True) * 100

    print("=" * 60)
    print("EXPLORACIÓN DE LA COLUMNA CONDICIÓN")
    print("=" * 60)
    print(f"Número de opciones distintas encontradas: {total_opciones}\n")
    print("Desglose de registros por categoría:")
    print("-" * 60)
    for opcion, cantidad in conteo_condicion.items():
        nombre_opcion = "Valores vacíos (NaN)" if pd.isna(opcion) else opcion
        porcentaje = porcentaje_condicion[opcion]
        print(f" -> {nombre_opcion:<25} | {cantidad:>6,} anuncios ({porcentaje:.2f}%)")
    print("=" * 60)


✅ Columna de condición detectada: 'condition'
EXPLORACIÓN DE LA COLUMNA CONDICIÓN
Número de opciones distintas encontradas: 7

Desglose de registros por categoría:
------------------------------------------------------------
 -> Valores vacíos (NaN)      | 212,922 anuncios (74.84%)
 -> Very good                 | 36,136 anuncios (12.70%)
 -> Unworn                    | 14,671 anuncios (5.16%)
 -> New                       | 11,139 anuncios (3.92%)
 -> Good                      |  8,251 anuncios (2.90%)
 -> Fair                      |  1,324 anuncios (0.47%)
 -> Poor                      |     45 anuncios (0.02%)
 -> Incomplete                |      3 anuncios (0.00%)


In [55]:
# ============================================================
# SEGMENTACIÓN: FILTRADO DE RELOJES NUEVOS (MERCADO OBJETIVO)
# ============================================================

condiciones_nuevas = ['New', 'Unworn']

# ── PASO 1: eliminar columnas completamente vacías del CSV ──
cols_vacias = [c for c in df.columns if df[c].isna().all()]
if cols_vacias:
    print(f"🗑️  Columnas completamente vacías eliminadas del CSV: {cols_vacias}")
    print("   (Estas columnas no aportan información al análisis)")
df_limpio = df.drop(columns=cols_vacias)

# ── PASO 2: detectar la columna de condición real ──
# En este CSV existen dos columnas parecidas: 'cond' y 'condition'.
# 'condition' contiene valores desplazados/incompletos en parte del scraping;
# si se usa para filtrar, selecciona filas donde 'name', 'cond' y 'sex' quedan vacíos.
# Por eso elegimos la columna que contiene más registros New/Unworn válidos.
condition_candidates = [c for c in ['cond', 'condition'] if c in df_limpio.columns]
condition_scores = {
    c: int(df_limpio[c].isin(condiciones_nuevas).sum())
    for c in condition_candidates
}

col_cond = None
if condition_scores:
    mejor_columna = max(condition_scores, key=condition_scores.get)
    if condition_scores[mejor_columna] > 0:
        col_cond = mejor_columna

print("Candidatas de condición detectadas:", condition_scores)
print(f"Columna usada para filtrar condición: {col_cond}")

# ── PASO 3: filtrar relojes nuevos ──
if col_cond:
    df_nuevos = df_limpio[df_limpio[col_cond].isin(condiciones_nuevas)].copy()

    # Normalizar el nombre a 'cond' sin crear columnas duplicadas.
    if col_cond != 'cond':
        df_nuevos['cond'] = df_nuevos[col_cond]

    # Eliminamos la columna alternativa no usada para evitar diagnósticos confusos de nulos.
    columnas_condicion_extra = [c for c in ['condition'] if c in df_nuevos.columns and c != col_cond]
    if columnas_condicion_extra:
        df_nuevos = df_nuevos.drop(columns=columnas_condicion_extra)
else:
    print("⚠️  Sin datos de condición — se usan todos los registros.")
    df_nuevos = df_limpio.copy()
    df_nuevos['cond'] = np.nan

print("=" * 60)
print("FILTRADO COMPLETADO CON ÉXITO 🎯")
print("=" * 60)
print(f"Registros en el dataset original: {df.shape[0]:>8,}")
print(f"Registros en el dataset de NUEVOS:{df_nuevos.shape[0]:>8,}")
print(f"Porcentaje de datos retenido:     {df_nuevos.shape[0]/df.shape[0]*100:>7.2f}%")
print("-" * 60)
print("Distribución final en el dataset de campaña:")
if 'cond' in df_nuevos.columns:
    print(df_nuevos['cond'].value_counts())
print("=" * 60)


Candidatas de condición detectadas: {'cond': 101829, 'condition': 25810}
Columna usada para filtrar condición: cond
FILTRADO COMPLETADO CON ÉXITO 🎯
Registros en el dataset original:  284,491
Registros en el dataset de NUEVOS: 101,829
Porcentaje de datos retenido:       35.79%
------------------------------------------------------------
Distribución final en el dataset de campaña:
cond
New       66198
Unworn    35631
Name: count, dtype: int64


### 3.3. Comprobamos las variables numéricas y categóricas del dataset.

> 📌 **A partir de aquí, todo el cuaderno trabaja sobre `df_nuevos`** (solo relojes en condición *New* o *Unworn*), tal y como se definió en el filtrado de la sección anterior. El `df` original (con relojes usados) queda disponible por si se necesita una comparativa, pero no se usa en el resto de este cuaderno.

In [56]:
# Separamos las columnas por tipo de datos
variables_numericas = df_nuevos.select_dtypes(include=[np.number]).columns.tolist()
variables_categoricas = df_nuevos.select_dtypes(include=['object', 'category']).columns.tolist()

print("="*60)
print("RESUMEN DE TIPOS DE VARIABLES")
print("="*60)
print(f"Variables Numéricas ({len(variables_numericas)}):")
print(f" -> {variables_numericas}\n")

print(f"Variables Categóricas ({len(variables_categoricas)}):")
print(f" -> {variables_categoricas}")
print("="*60)

RESUMEN DE TIPOS DE VARIABLES
Variables Numéricas (3):
 -> ['Unnamed: 0', 'price', 'yop']

Variables Categóricas (12):
 -> ['name', 'brand', 'model', 'ref', 'mvmt', 'casem', 'bracem', 'cond', 'sex', 'size', 'yop_raw', 'yop_fuente']


## 4. Mapeo y normalización de columnas

⚙️ **Paso importante.** Revisamos la lista de columnas impresa arriba y comparamos con el diccionario `COLUMN_MAP` de la siguiente celda. Solo se renombrarán las columnas que existan realmente en tu versión del dataset — así el resto del cuaderno funciona aunque algún nombre cambie ligeramente.

Esto convierte nombres técnicos del CSV en nombres de negocio consistentes que usaremos durante todo el proyecto (y que después usarás en los títulos y filtros del dashboard).


In [57]:
# ============================================================
# NORMALIZACIÓN DE NOMBRES DE COLUMNA
# ============================================================
# El CSV de Chrono24 usa nombres abreviados (casem, bracem, ref).
# Los renombramos a nombres descriptivos para el dashboard.
# Solo se renombran las columnas que existan en la versión del
# dataset descargada → el código funciona con cualquier versión.

COLUMN_MAP = {
    'ref'   : 'reference',
    'casem' : 'case_material',
    'bracem': 'bracelet_material',
    'size'  : 'case_size',
}

existing_map = {k: v for k, v in COLUMN_MAP.items() if k in df_nuevos.columns}
df_nuevos = df_nuevos.rename(columns=existing_map)

print("Columnas normalizadas:")
for original, nuevo in existing_map.items():
    print(f"  '{original}'  →  '{nuevo}'")
print(f"\nColumnas finales disponibles en df_nuevos:")
print(df_nuevos.columns.tolist())


Columnas normalizadas:
  'ref'  →  'reference'
  'casem'  →  'case_material'
  'bracem'  →  'bracelet_material'
  'size'  →  'case_size'

Columnas finales disponibles en df_nuevos:
['Unnamed: 0', 'name', 'price', 'brand', 'model', 'reference', 'mvmt', 'case_material', 'bracelet_material', 'yop', 'cond', 'sex', 'case_size', 'yop_raw', 'yop_fuente']


### 4.1. Recuperación conservadora de campos técnicos

Antes de aceptar como definitivos los nulos de `mvmt`, `case_material` y `bracelet_material`, comprobamos si hay referencias repetidas en el marketplace. Si otros anuncios con la misma combinación `brand + reference` informan de forma consistente el mismo valor, lo reutilizamos para completar el registro incompleto.

Esta estrategia no inventa materiales por marca o por modelo genérico: solo rellena cuando la referencia exacta tiene consenso suficiente. Además, se crean columnas `_fuente` para distinguir valores originales, imputados por referencia y valores todavía sin dato.


In [58]:
# ============================================================
# RECUPERACIÓN CONSERVADORA DE CAMPOS TÉCNICOS
# ============================================================

columnas_tecnicas = ['mvmt', 'case_material', 'bracelet_material']

# Normalizamos cadenas vacías y espacios en blanco como nulos reales.
for col in columnas_tecnicas + ['brand', 'reference', 'model', 'name']:
    if col in df_nuevos.columns:
        df_nuevos[col] = (
            df_nuevos[col]
            .astype('string')
            .str.strip()
            .replace('', pd.NA)
        )


def rellenar_por_referencia(df, col, keys=('brand', 'reference'), min_obs=2, min_share=0.85):
    """Rellena nulos usando consenso de otros anuncios con la misma referencia.

    min_obs exige al menos dos anuncios informados para esa referencia.
    min_share evita rellenar referencias con materiales o movimientos ambiguos.
    """
    keys = [k for k in keys if k in df.columns]
    if col not in df.columns or len(keys) != 2:
        return df, 0, 0

    df[f'{col}_fuente'] = np.where(df[col].notna(), 'original', 'sin_dato')

    base = df[df[keys].notna().all(axis=1) & df[col].notna()]
    conteos = (
        base
        .groupby(keys)[col]
        .value_counts(dropna=True)
        .rename('n')
        .reset_index()
    )
    if conteos.empty:
        return df, 0, 0

    conteos['total_ref'] = conteos.groupby(keys)['n'].transform('sum')
    conteos['share'] = conteos['n'] / conteos['total_ref']

    reglas = (
        conteos[(conteos['n'] >= min_obs) & (conteos['share'] >= min_share)]
        .sort_values(keys + ['share', 'n'], ascending=[True, True, False, False])
        .drop_duplicates(keys)
    )
    mapa = reglas.set_index(keys)[col]

    mask = df[col].isna() & df[keys].notna().all(axis=1)
    idx = pd.MultiIndex.from_frame(df.loc[mask, keys])
    valores = pd.Series(idx.map(mapa), index=df.index[mask], dtype='string')
    mask_relleno = mask.copy()
    mask_relleno.loc[mask] = valores.notna().to_numpy()

    df.loc[mask_relleno, col] = valores.dropna().to_numpy()
    df.loc[mask_relleno, f'{col}_fuente'] = 'imputado_por_referencia'

    return df, int(mask_relleno.sum()), int(len(reglas))


resumen_recuperacion = []
for col in columnas_tecnicas:
    nulos_antes = int(df_nuevos[col].isna().sum())
    df_nuevos, recuperados, reglas = rellenar_por_referencia(df_nuevos, col)
    nulos_despues = int(df_nuevos[col].isna().sum())
    resumen_recuperacion.append({
        'columna': col,
        'nulos_antes': nulos_antes,
        'recuperados': recuperados,
        'nulos_despues': nulos_despues,
        'reglas_referencia': reglas,
    })


def inferir_movimiento_desde_nombre(nombre):
    if pd.isna(nombre):
        return pd.NA
    texto = str(nombre).lower()
    candidatos = []
    if re.search(r'\bautomatic\b|\bautomático\b', texto):
        candidatos.append('Automatic')
    if re.search(r'\bquartz\b|\bcuarzo\b', texto):
        candidatos.append('Quartz')
    if re.search(r'manual winding|manual-winding|hand[- ]?wound|\bmanual\b', texto):
        candidatos.append('Manual winding')
    candidatos = list(dict.fromkeys(candidatos))
    return candidatos[0] if len(candidatos) == 1 else pd.NA


if {'mvmt', 'name'}.issubset(df_nuevos.columns):
    mvmt_inferido = df_nuevos['name'].apply(inferir_movimiento_desde_nombre).astype('string')
    mask_mvmt_nombre = df_nuevos['mvmt'].isna() & mvmt_inferido.notna()
    df_nuevos.loc[mask_mvmt_nombre, 'mvmt'] = mvmt_inferido[mask_mvmt_nombre]
    df_nuevos.loc[mask_mvmt_nombre, 'mvmt_fuente'] = 'inferido_desde_nombre'

    resumen_recuperacion.append({
        'columna': 'mvmt',
        'nulos_antes': int(mask_mvmt_nombre.sum() + df_nuevos['mvmt'].isna().sum()),
        'recuperados': int(mask_mvmt_nombre.sum()),
        'nulos_despues': int(df_nuevos['mvmt'].isna().sum()),
        'reglas_referencia': 'regex_nombre',
    })


resumen_recuperacion = pd.DataFrame(resumen_recuperacion)
print("Recuperación de campos técnicos:")
print(resumen_recuperacion)

print("\nDistribución de fuentes tras recuperación:")
for col in columnas_tecnicas:
    print(f"\n{col}:")
    print(df_nuevos[f'{col}_fuente'].value_counts(dropna=False))


Recuperación de campos técnicos:
             columna  nulos_antes  recuperados  nulos_despues  \
0               mvmt        55430        34641          20789   
1      case_material        57590        33291          24299   
2  bracelet_material        61441        28338          33103   
3               mvmt        20789         3431          17358   

  reglas_referencia  
0              7597  
1              6865  
2              5589  
3      regex_nombre  

Distribución de fuentes tras recuperación:

mvmt:
mvmt_fuente
original                   46399
imputado_por_referencia    34641
sin_dato                   17358
inferido_desde_nombre       3431
Name: count, dtype: int64

case_material:
case_material_fuente
original                   44239
imputado_por_referencia    33291
sin_dato                   24299
Name: count, dtype: int64

bracelet_material:
bracelet_material_fuente
original                   40388
sin_dato                   33103
imputado_por_referencia    28338
Name

## 5. Calidad del dato: nulos y duplicados (Checklist I)

Cuantificamos los valores nulos y los registros duplicados **antes** de decidir qué estrategia de limpieza aplicar.


In [59]:
# ============================================================
# DIAGNÓSTICO: COLUMNAS CON NULOS RELEVANTES
# ============================================================
# La limpieza de columnas vacías y el renombrado ya se hicieron
# en la Sección 3.2. Aquí solo documentamos el estado actual.

print("Estado del DataFrame de campaña (df_nuevos):")
print(f"  Filas: {df_nuevos.shape[0]:,}  |  Columnas: {df_nuevos.shape[1]}")
print()

nulos = df_nuevos.isnull().sum()
pct   = (nulos / len(df_nuevos) * 100).round(1)
resumen = pd.DataFrame({'nulos': nulos, 'pct_%': pct})
resumen = resumen[resumen['nulos'] > 0].sort_values('pct_%', ascending=False)

if resumen.empty:
    print("✅ No se detectaron valores nulos en las columnas de interés.")
else:
    print("Columnas con valores nulos (documentado para gobernanza):")
    print(resumen)
    print()
    print("📌 Estrategia aplicada:")
    print("  · Columnas COMPLETAMENTE vacías → eliminadas en Sección 3.2")
    print("  · Columnas PARCIALMENTE vacías (ej. case_material, yop) →")
    print("    se mantienen; los gráficos las usan solo cuando hay dato válido.")

# Filas duplicadas
dups = df_nuevos.duplicated().sum()
print(f"\nFilas completamente duplicadas: {dups:,} ({dups/len(df_nuevos)*100:.2f}%)")


Estado del DataFrame de campaña (df_nuevos):
  Filas: 101,829  |  Columnas: 18

Columnas con valores nulos (documentado para gobernanza):
                   nulos  pct_%
yop                33254  32.70
bracelet_material  33103  32.50
case_material      24299  23.90
mvmt               17358  17.00
case_size          11639  11.40
reference           9652   9.50
sex                 9425   9.30
price               8475   8.30
model               7908   7.80

📌 Estrategia aplicada:
  · Columnas COMPLETAMENTE vacías → eliminadas en Sección 3.2
  · Columnas PARCIALMENTE vacías (ej. case_material, yop) →
    se mantienen; los gráficos las usan solo cuando hay dato válido.

Filas completamente duplicadas: 0 (0.00%)


In [60]:
# Valores nulos por columna
nulos = df_nuevos.isnull().sum()
porcentaje_nulos = (nulos / len(df_nuevos) * 100).round(2)

resumen_nulos = pd.DataFrame({'nulos': nulos, 'pct_nulos': porcentaje_nulos})
resumen_nulos = resumen_nulos[resumen_nulos['nulos'] > 0].sort_values('pct_nulos', ascending=False)

print("Valores nulos por columna:")
print(resumen_nulos if not resumen_nulos.empty else "No se detectaron valores nulos.")

# Filas duplicadas
duplicados = df_nuevos.duplicated().sum()
print(f"\nFilas completamente duplicadas: {duplicados:,} ({duplicados/len(df_nuevos)*100:.2f}% del total)")

Valores nulos por columna:
                   nulos  pct_nulos
yop                33254      32.66
bracelet_material  33103      32.51
case_material      24299      23.86
mvmt               17358      17.05
case_size          11639      11.43
reference           9652       9.48
sex                 9425       9.26
price               8475       8.32
model               7908       7.77

Filas completamente duplicadas: 0 (0.00% del total)


### 5.1 Estrategia de limpieza (justificación)

Con base en el resultado anterior, aplicamos las siguientes reglas. **Ajusta los porcentajes/umbrales si tu resultado es muy distinto** — la justificación debe basarse en los números reales que obtengas:

1. **Columnas con un % de nulos muy alto (p. ej. > 60%)**: se documentan pero no se eliminan en este paso; se evalúa su utilidad real para el dashboard antes de descartarlas definitivamente.
2. **Filas duplicadas exactas**: se eliminan, ya que representan el mismo anuncio repetido y distorsionarían las métricas de "cuota de mercado por marca".
3. **Precio**: es la variable más crítica del proyecto. Las filas sin un precio válido (nulo, texto no numérico o ≤ 0) se eliminan, porque no aportan información a ningún análisis de precio/segmento.
4. **Outliers extremos de precio**: se recorta el 0.5% superior de la distribución. Esto evita que un puñado de piezas de coleccionista extremadamente caras (p. ej. ediciones únicas de Richard Mille) distorsionen las medias y los gráficos del resto de marcas. Se documenta cuántas filas se eliminan por este motivo.


In [61]:
def limpiar_precio(valor):
    # Convierte un precio en formato texto (ej. '$12,500.00', 'CHF 9.800') a float.
    # Devuelve np.nan si no se puede convertir.
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)
    texto = str(valor)
    # Nos quedamos solo con dígitos, comas y puntos
    texto_limpio = re.sub(r'[^\d.,]', '', texto)
    # Asumimos formato anglosajón: la coma es separador de miles
    texto_limpio = texto_limpio.replace(',', '')
    try:
        return float(texto_limpio) if texto_limpio else np.nan
    except ValueError:
        return np.nan


# Creamos una columna de precio limpio y numérico (SOBREESCRIBIENDO 'price')
columna_precio_origen = 'price_raw' if 'price_raw' in df_nuevos.columns else 'price'

# REEMPLAZO AQUÍ: Cambiamos df_nuevos['price_usd'] por df_nuevos['price']
df_nuevos['price'] = df_nuevos[columna_precio_origen].apply(limpiar_precio).astype('Float64')

# Actualizamos también los prints para que apunten a 'price'
print(f"Precios convertidos correctamente: {df_nuevos['price'].notna().sum():,} de {len(df_nuevos):,} filas")
print("\nEstadísticos iniciales de precio (sin limpiar outliers todavía):")
print(df_nuevos['price'].describe())

Precios convertidos correctamente: 93,354 de 101,829 filas

Estadísticos iniciales de precio (sin limpiar outliers todavía):
count      93,354.00
mean       20,286.94
std        96,011.34
min            26.00
25%         2,299.00
50%         5,797.00
75%        11,895.00
max     9,000,000.00
Name: price, dtype: Float64


In [62]:
filas_antes = len(df_nuevos)

# 1. Eliminamos duplicados exactos
df_nuevos = df_nuevos.drop_duplicates()
filas_tras_duplicados = len(df_nuevos)

# 2. Eliminamos filas sin precio válido
df_nuevos = df_nuevos[df_nuevos['price'].notna() & (df_nuevos['price'] > 0)]
filas_tras_precio = len(df_nuevos)

# 3. Recortamos el 0.5% superior de precios (outliers extremos)
limite_superior = df_nuevos['price'].quantile(0.995)
df_nuevos = df_nuevos[df_nuevos['price'] <= limite_superior]
filas_finales = len(df_nuevos)

print("RESUMEN DE LIMPIEZA")
print("-" * 40)
print(f"Filas iniciales:                    {filas_antes:,}")
print(f"Tras eliminar duplicados:            {filas_tras_duplicados:,}  (-{filas_antes - filas_tras_duplicados:,})")
print(f"Tras eliminar precios inválidos:     {filas_tras_precio:,}  (-{filas_tras_duplicados - filas_tras_precio:,})")
print(f"Tras recortar outliers (P99.5):      {filas_finales:,}  (-{filas_tras_precio - filas_finales:,})")
print(f"\nLímite superior de precio aplicado: ${limite_superior:,.0f}")
print(f"Total de filas eliminadas:           {filas_antes - filas_finales:,} ({(filas_antes-filas_finales)/filas_antes*100:.2f}%)")

RESUMEN DE LIMPIEZA
----------------------------------------
Filas iniciales:                    101,829
Tras eliminar duplicados:            101,829  (-0)
Tras eliminar precios inválidos:     93,354  (-8,475)
Tras recortar outliers (P99.5):      92,892  (-462)

Límite superior de precio aplicado: $375,000
Total de filas eliminadas:           8,937 (8.78%)


## 6. Variable de negocio: "Tier de Lujo"

Creamos una variable categórica de negocio a partir del precio. Esta variable será uno de los **filtros de segmentación** del dashboard, y es clave para la narrativa: nos permite recomendar qué marcas/tiers encajan con cada tipo de zona VIP (p. ej. un *lounge* general vs. un palco ultra-exclusivo).

⚙️ Los umbrales son un punto de partida. Tras ver la distribución real de `price_usd`, ajústalos si lo consideras necesario (por ejemplo, basándote en los percentiles 25/50/90).


In [63]:
def asignar_tier(precio):
    if precio < 2000:
        return '1. Entrada'
    elif precio < 10000:
        return '2. Premium'
    elif precio < 50000:
        return '3. Alta Gama'
    else:
        return '4. Ultra-Lujo'

df_nuevos['tier_lujo'] = df_nuevos['price'].apply(asignar_tier)

resumen_tier = df_nuevos['tier_lujo'].value_counts().sort_index()
resumen_tier_pct = (resumen_tier / len(df_nuevos) * 100).round(1)

print("Distribución de modelos por Tier de Lujo:")
for tier, n in resumen_tier.items():
    print(f"  {tier:<15} {n:>8,} anuncios  ({resumen_tier_pct[tier]}%)")

Distribución de modelos por Tier de Lujo:
  1. Entrada        20,123 anuncios  (21.7%)
  2. Premium        45,603 anuncios  (49.1%)
  3. Alta Gama      20,126 anuncios  (21.7%)
  4. Ultra-Lujo      7,040 anuncios  (7.6%)


## 7. Estadística descriptiva por marca (Checklist II)

Calculamos medidas de tendencia central (media, mediana) y de dispersión (desviación estándar, rango) del precio, agrupadas por marca. Esta tabla es la base numérica de los gráficos y de las conclusiones de negocio.


In [64]:
top_n_marcas = 10
marcas_top = df_nuevos['brand'].value_counts().head(top_n_marcas).index

resumen_marca = (
    df_nuevos[df_nuevos['brand'].isin(marcas_top)]
    .groupby('brand')['price']
    .agg(
        n_modelos='count',
        precio_medio='mean',
        precio_mediana='median',
        desviacion_std='std',
        precio_min='min',
        precio_max='max'
    )
    .sort_values('precio_medio', ascending=False)
)

resumen_marca

,n_modelos,precio_medio,precio_mediana,desviacion_std,precio_min,precio_max
brand,,,,,,
Patek Philippe,3992,"108,286.10","88,196.50","71,062.66","5,789.00","375,000.00"
Audemars Piguet,3902,"90,175.24","63,939.00","70,675.58","2,869.00","375,000.00"
Hublot,8045,"20,602.89","17,004.00","16,543.55",690.00,"279,995.00"
IWC,3768,"11,815.36","8,909.00","13,385.74","1,036.00","229,064.00"
Omega,16445,"9,062.67","6,955.00","7,654.80",399.00,"237,142.00"
Breitling,8201,"7,047.40","5,808.00","4,515.45",150.00,"64,514.00"
Tudor,3769,"4,324.66","4,172.00","2,146.19",966.00,"80,506.00"
TAG Heuer,6422,"3,740.40","2,990.00","3,330.84",230.00,"74,346.00"
Longines,10789,"2,152.78","1,902.00","1,243.10",172.00,"17,765.00"


## 8. Matriz de correlación (Checklist II)

Analizamos cómo se relacionan entre sí las variables numéricas disponibles. Esto nos ayuda a identificar, por ejemplo, si el año de producción influye en el precio de venta.



In [65]:
columnas_numericas = df_nuevos.select_dtypes(include=np.number).columns.tolist()
print("Columnas numéricas disponibles para el análisis de correlación:")
print(columnas_numericas)

if len(columnas_numericas) >= 2:
    matriz_corr = df_nuevos[columnas_numericas].corr(numeric_only=True)

    fig_corr = px.imshow(
        matriz_corr,
        text_auto='.2f',
        color_continuous_scale='RdBu_r',
        title='Matriz de correlación — variables numéricas del dataset de relojes de lujo',
        aspect='auto'
    )
    fig_corr.update_layout(template='plotly_white')
    fig_corr.show()
else:
    print("\nSolo hay una variable numérica disponible (price).")
    print("No es posible calcular una matriz de correlación con una sola variable.")
    print("Alternativa: se puede analizar 'price' frente a variables categóricas (marca, material, condición).")

Columnas numéricas disponibles para el análisis de correlación:
['Unnamed: 0', 'price', 'yop']


### 📊 Interpretación de la Matriz de Correlación

> 💡 **Clave de negocio:** esta sección responde a ¿hay alguna variable que "prediga" el precio de un reloj de lujo nuevo?

La matriz muestra el **coeficiente de correlación de Pearson** (valores entre −1 y +1) entre todas las variables numéricas del dataset:

- **+1** → relación perfectamente positiva (cuando una sube, la otra también sube al mismo ritmo)
- **0** → sin relación lineal detectable
- **−1** → relación perfectamente negativa (cuando una sube, la otra baja)

#### Qué buscar en los resultados

| Par de variables | Correlación esperada | Interpretación de negocio |
|---|---|---|
| `price` ↔ `yop` (año de fabricación) | Moderada positiva (0,2–0,5) | Los modelos más recientes tienden a costar más → **apuesta por catálogo actual** |
| `price` ↔ `Unnamed: 0` (índice del CSV) | ~0 | El orden en que el anuncio fue scrapeado no afecta al precio — sin relevancia de negocio |
| `yop` ↔ `Unnamed: 0` | Posiblemente positiva | Los anuncios más recientes en el CSV tienden a ser de modelos más nuevos — artefacto del scraping, no de negocio |

#### Lectura práctica para la decisión del stand VIP

- Si `price` y `yop` tienen correlación **> 0,3**: el año de fabricación es un criterio válido de selección — en el stand se deben priorizar modelos de los últimos 3-5 años.
- Si la correlación `price` ↔ `yop` es **cercana a 0**: el precio está determinado principalmente por la **marca y la referencia concreta**, no por la antigüedad. En ese caso, la estrategia del stand debe pivotar hacia la narrativa de marca, no hacia "modelos recientes".

> ⚠️ **Limitación importante:** con solo 2-3 variables numéricas disponibles, la matriz de correlación tiene utilidad descriptiva limitada. El análisis más potente para la decisión de negocio está en los Gráficos 1-4, que combinan dimensiones numéricas y categóricas de forma visual e interactiva.


---
# 🟩 — Storytelling visual: 4 gráficos interactivos (Checklist IV)

Cada gráfico se diseña para responder **una pregunta de negocio concreta**, con títulos y ejes en lenguaje natural (sin tecnicismos de código), tal y como exige el checklist de visualización e interactividad.


## Gráfico 1 — ¿Cómo se distribuyen los precios dentro de cada marca?

**Pregunta de negocio:** ¿Qué marcas tienen mayor variabilidad de precios (es decir, abarcan desde piezas accesibles hasta ultra-exclusivas) y cuáles son más homogéneas? Esto ayuda a decidir qué marca encaja mejor con cada tipo de zona VIP.


In [66]:
marcas_top12 = df_nuevos['brand'].value_counts().head(12).index
df_top = df_nuevos[df_nuevos['brand'].isin(marcas_top12)]

fig1 = px.box(
    df_top,
    x='brand',
    y='price',
    color='brand',
    log_y=True,   # Escala logarítmica: expande visualmente las marcas
                  # de precio más bajo (Tudor, Omega) sin ocultar
                  # las de precio alto (Patek, Audemars Piguet)
    title='Distribución de precios por marca de relojería de lujo (escala logarítmica)',
    labels={'brand': '', 'price': 'Precio de venta — USD (escala log)'},
    points=False,
    width=1100,
    height=650
)

fig1.update_layout(
    template='plotly_white',
    showlegend=False,
    xaxis_tickangle=-45,
    annotations=[dict(
        text="⚠️ Escala logarítmica: cada división multiplica el precio por 10. "
             "Permite comparar visualmente marcas con rangos de precio muy distintos.",
        xref="paper", yref="paper",
        x=0, y=-0.18, showarrow=False,
        font=dict(size=11, color="gray")
    )]
)

fig1.show()


### 📝 Conclusiones — Gráfico 1: Distribución de precios por marca

> 💡 **Clave de negocio — ¿Qué marca elige para el stand VIP?**
>
> Identifica en el gráfico las marcas según estas tres categorías estratégicas:
>
> | Perfil en el gráfico | Qué comunica al público VIP | Recomendación para el stand |
> |---|---|---|
> | **Caja alta y estrecha** (rango intercuartílico pequeño) | Exclusividad homogénea — cada pieza transmite el mismo mensaje de precio | Ideal para stands *monotemáticos* de ultra-lujo (palcos privados, lounges de máximo nivel) |
> | **Caja ancha que abarca varios tiers** | Versatilidad — la marca llega desde el público VIP general hasta el ultra-exclusivo | Ideal para stands que deben funcionar en *distintas* zonas VIP de un mismo estadio |
> | **Mediana baja, con atípicos muy altos** | Marca reconocible y accesible, con ediciones premium para coleccionistas | Útil como "gancho de volumen" en el lounge general; combínala con una segunda marca ultra-premium |

**Por qué la escala logarítmica.** El gráfico usa escala log para que marcas como Tudor u Omega (cuyos precios se concentran entre $1.000 y $8.000) sean visualmente comparables con Patek Philippe o Audemars Piguet (cuyos precios pueden superar los $50.000). En escala lineal, las marcas de precio más bajo aparecen aplastadas y su variabilidad queda oculta — algo especialmente engañoso para tomar decisiones de portfolio.

**Sesgos a vigilar.**
- Algunas marcas pueden tener pocas observaciones tras filtrar por "solo nuevos" → su caja no es estadísticamente robusta. Cruza siempre con el Gráfico 2 (volumen de anuncios).



## Gráfico 2 — ¿Qué marcas tienen mayor número de anuncios en el dataset?

**Pregunta de negocio:** El volumen de anuncio indica la representación de la marca en el mercado global y puede hacer una idea de su reconocimiento para un publico general y no solo del ambito de la exclusividad


In [67]:
conteo_marcas = df_nuevos['brand'].value_counts().head(15).reset_index()
conteo_marcas.columns = ['marca', 'numero_anuncios']

fig2 = px.bar(
    conteo_marcas,
    x='numero_anuncios',
    y='marca',
    orientation='h',
    title='¿Qué marcas de relojes de lujo tienen mayor presencia en el mercado?',
    labels={'numero_anuncios': 'Número de anuncios activos', 'marca': ''},
    text='numero_anuncios'
)
fig2.update_layout(
    template='plotly_white',
    yaxis={'categoryorder': 'total ascending'}
)
fig2.show()

### 📝 Conclusiones — Gráfico 2: Volumen de presencia por marca

> 💡 **Clave de negocio — ¿Cuánta tracción de mercado tiene cada candidata?**
>
> El volumen de anuncios de relojes **nuevos** disponibles es el mejor proxy público del *apetito real de mercado* por cada marca, mucho más fiable que el historial de colecciones o el prestigio percibido.
>
> - **Las 3-4 marcas con más anuncios** = reconocibles para el 80% del público VIP del Mundial, incluso sin ser aficionados a la relojería. Son la opción más segura para el *lounge general* o la zona de networking del estadio.
> - **Las marcas en posiciones intermedias** (5ª-10ª) = perfil aspiracional. Conocidas por los entusiastas; atraen la mirada del público VIP sin resultar "demasiado comerciales". Son ideales para zonas de acceso ligeramente más exclusivo.
> - **Las marcas con pocos anuncios pero precio alto** (ver Gráfico 1) = ultra-nicho. Perfectas para un *palco privado* con invitados de primer nivel, donde el anfitrión quiere diferenciarse.

**Criterio de selección cruzado (Gráficos 1 + 2):**
Una marca en el **top 5 de este gráfico + mediana de precio alta en el Gráfico 1** es la combinación óptima para el stand principal. Evita marcas con muchos anuncios pero precio medio bajo — comunican popularidad más que exclusividad.

**Sesgos a vigilar.**
- El volumen de Chrono24 refleja la oferta de los *revendedores*, no las ventas directas de las marcas. Una casa como Patek Philippe distribuye casi todo su stock a través de boutiques autorizadas, por lo que puede aparecer con menos anuncios de lo que su cuota de mercado real implicaría.
- Sesgo geográfico: vendedores concentrados en Europa y EE.UU. → marcas fuertes en Asia pueden estar subrepresentadas.


## Gráfico 3 — Mapa de oportunidad: ¿qué marcas combinan catálogo reciente y posicionamiento premium?

**Pregunta de negocio:** de las marcas de relojería de lujo (ninguna patrocinadora oficial del Mundial 2026), ¿cuáles ofrecen actualmente un catálogo de modelos **recientes** (no "stock antiguo") a un **precio medio elevado**, y además tienen suficiente presencia en el mercado para ser reconocibles por un público VIP general?

Construimos un mapa de posicionamiento: cada burbuja es una marca. El eje horizontal es el **año medio de fabricación** de sus modelos nuevos disponibles (a mayor valor, catálogo más actual); el eje vertical es el **precio medio**; el tamaño de la burbuja es el **número de modelos disponibles** (visibilidad/reconocimiento). Las líneas discontinuas marcan las medianas globales y dividen el mapa en 4 cuadrantes.

In [68]:
# Agregamos por marca: año medio de fabricación, precio medio y nº de modelos
df_yop_valido = df_nuevos.dropna(subset=['yop', 'price'])

resumen_oportunidad = (
    df_yop_valido
    .groupby('brand')
    .agg(
        anio_medio=('yop', 'mean'),
        precio_medio=('price', 'mean'),
        n_modelos=('price', 'count')
    )
    .reset_index()
)

# Filtramos marcas con muy poca presencia para evitar burbujas poco fiables
MIN_MODELOS = 5
resumen_oportunidad = resumen_oportunidad[resumen_oportunidad['n_modelos'] >= MIN_MODELOS]

mediana_anio = resumen_oportunidad['anio_medio'].median()
mediana_precio = resumen_oportunidad['precio_medio'].median()

fig3 = px.scatter(
    resumen_oportunidad,
    x='anio_medio',
    y='precio_medio',
    size='n_modelos',
    color='brand',
    text='brand',
    title='¿Qué marcas combinan catálogo reciente y precio premium? (tamaño = nº de modelos nuevos disponibles)',
    labels={'anio_medio': 'Año medio de fabricación de los modelos', 'precio_medio': 'Precio medio (USD)', 'brand': 'Marca'},
    size_max=50
)

fig3.update_traces(textposition='top center')

# Líneas de referencia: medianas globales -> dividen el mapa en 4 cuadrantes
fig3.add_vline(x=mediana_anio, line_dash='dash', line_color='gray')
fig3.add_hline(y=mediana_precio, line_dash='dash', line_color='gray')

fig3.update_layout(template='plotly_white', showlegend=False, width=1100, height=650)
fig3.show()

print(f"Mediana del año medio de fabricación: {mediana_anio:.1f}")
print(f"Mediana del precio medio: ${mediana_precio:,.0f}")

print("\nMarcas en el cuadrante 'oportunidad' (catálogo reciente + precio alto):")
oportunidad = resumen_oportunidad[
    (resumen_oportunidad['anio_medio'] >= mediana_anio) &
    (resumen_oportunidad['precio_medio'] >= mediana_precio)
].sort_values('precio_medio', ascending=False)
print(oportunidad[['brand', 'anio_medio', 'precio_medio', 'n_modelos']])

Mediana del año medio de fabricación: 2021.0
Mediana del precio medio: $13,034

Marcas en el cuadrante 'oportunidad' (catálogo reciente + precio alto):
               brand  anio_medio  precio_medio  n_modelos
5             Hublot    2,022.71     20,652.70       5893
7   Jaeger-LeCoultre    2,020.95     17,343.55       1249
12           Panerai    2,021.52     13,034.23       2028


### 📝 Conclusiones — Gráfico 3: Mapa de oportunidad (catálogo reciente × precio)

> 💡 **Clave de negocio — Shortlist final de candidatas para el stand VIP**
>
> El **cuadrante superior derecho** (catálogo reciente + precio medio elevado) es tu zona de decisión. Las marcas que aparecen allí cumplen los tres criterios simultáneamente:
>
> 1. **Actualidad** → sus modelos disponibles son recientes, no "stock sobrante" de años pasados.
> 2. **Posicionamiento premium** → su precio medio supera la mediana del mercado de lujo.
> 3. **Presencia suficiente** → tamaño de burbuja grande = catálogo variado → el stand tiene producto con qué nutrir diferentes puntos de precio dentro de la zona VIP.
>
> Las marcas del cuadrante **superior izquierdo** (precio alto, catálogo más antiguo) pueden complementar el portfolio con un mensaje de "herencia de marca" — útiles si el evento quiere proyectar tradición y atemporalidad, no "novedad 2026".

**Sesgos a vigilar.**
- El "año medio de fabricación" es un proxy de actualidad dentro del mercado de reventa, NO garantiza que la marca esté lanzando modelos nuevos en 2026. Antes de firmar el acuerdo, verifica el calendario de lanzamientos oficial de la marca seleccionada.
- Marcas con N < 5 anuncios válidos han sido excluidas del mapa para evitar burbujas no representativas.


## Gráfico 4 — ¿Qué rango de precios puede ofrecer cada marca candidata dentro de un stand VIP?

**Pregunta de negocio:** una vez identificadas las marcas con mejor posicionamiento (Gráfico 3), ¿qué **rango de precios** —y por tanto, qué tipo de pieza— puede ofrecer cada una? Una marca con modelos repartidos entre varios tiers de lujo puede anclar **distintos espacios VIP** (desde un lounge general hasta una sala privada ultra-exclusiva) sin salir de la misma marca; una marca concentrada en un único tier tiene un mensaje de marca más "puro" pero menos flexible.

Mostramos, para las marcas con mayor presencia en el dataset de relojes nuevos, cuántos modelos caen en cada Tier de Lujo (definido en la sección 6).

In [69]:
# Nos centramos en las marcas con mayor presencia (mismas que el Gráfico 2)
marcas_principales = df_nuevos['brand'].value_counts().head(10).index

df_tiers = df_nuevos[df_nuevos['brand'].isin(marcas_principales)]

conteo_tier_marca = (
    df_tiers
    .groupby(['brand', 'tier_lujo'])
    .size()
    .reset_index(name='n_modelos')
)

orden_tiers = ['1. Entrada', '2. Premium', '3. Alta Gama', '4. Ultra-Lujo']

fig4 = px.bar(
    conteo_tier_marca,
    x='brand',
    y='n_modelos',
    color='tier_lujo',
    category_orders={'tier_lujo': orden_tiers, 'brand': list(marcas_principales)},
    title='¿Qué rango de precios ofrece cada marca? (nº de modelos nuevos por Tier de Lujo)',
    labels={'brand': '', 'n_modelos': 'Número de modelos', 'tier_lujo': 'Tier de Lujo'},
)

fig4.update_layout(template='plotly_white', xaxis_tickangle=-30, width=1100, height=650, barmode='stack')
fig4.show()

# Tabla de apoyo: qué % de los modelos de cada marca caen en "Alta Gama" o "Ultra-Lujo"
tabla_pct = (
    conteo_tier_marca
    .pivot(index='brand', columns='tier_lujo', values='n_modelos')
    .fillna(0)
)
for tier in orden_tiers:
    if tier not in tabla_pct.columns:
        tabla_pct[tier] = 0

tabla_pct['total'] = tabla_pct[orden_tiers].sum(axis=1)
tabla_pct['% Alta Gama o superior'] = (
    (tabla_pct['3. Alta Gama'] + tabla_pct['4. Ultra-Lujo']) / tabla_pct['total'] * 100
).round(1)

tabla_pct.sort_values('% Alta Gama o superior', ascending=False)[['total', '% Alta Gama o superior']]

tier_lujo,total,% Alta Gama o superior
brand,,
Patek Philippe,"3,992.00",99.80
Audemars Piguet,"3,902.00",98.80
Hublot,"8,045.00",76.10
IWC,"3,768.00",40.70
Omega,"16,445.00",26.70
Breitling,"8,201.00",10.70
TAG Heuer,"6,422.00",2.00
Tudor,"3,769.00",1.20
Longines,"10,789.00",0.30


### 📝 Conclusiones — Gráfico 4: Cobertura de Tiers por marca

> 💡 **Clave de negocio — ¿Qué marca cubre todos los niveles del estadio VIP?**
>
> Un gran estadio del Mundial 2026 tendrá al menos tres niveles de zona VIP: el *lounge general* (accesible para todos los invitados VIP), los *palcos* (reducido, ejecutivos y patrocinadores corporativos) y las *suites privadas* (ultra-exclusivo, celebrities y directivos de primer nivel). Una sola marca debe poder cubrir los tres sin contradecirse.
>
> | Tipo de marca en este gráfico | Qué espacios puede cubrir | Ventaja para el stand |
> |---|---|---|
> | **Barra mayormente azul/verde** (Entrada + Premium) | Solo lounge general | Buen reconocimiento, bajo impacto VIP |
> | **Barra mayormente naranja/roja** (Alta Gama + Ultra-Lujo) | Solo palcos y suites | Impacto ultra-VIP, bajo reconocimiento masivo |
> | **Barra equilibrada en todos los colores** | Lounge + palcos + suites | **ÓPTIMA: una sola marca cubre todo el estadio** |
>
> Revisa la columna **"% Alta Gama o superior"** en la tabla: una marca con > 50% en Alta Gama  tiene suficiente profundidad premium para no "diluirse" en el lounge general, pero sí puede prescindir de piezas de Entrada si el público objetivo es netamente ejecutivo.

**Recomendación de portfolio definitiva (síntesis de los 4 gráficos):**
1. Selecciona del **cuadrante superior derecho del Gráfico 3** tu marca ancla.
2. Comprueba que tenga al menos un 30-40% de su catálogo en Alta Gama o Ultra-Lujo (**Gráfico 4**) y que su mediana de precio sea claramente premium (**Gráfico 1**).
3. Verifica que aparezca en el **top 8 del Gráfico 2** — si no, el público general del lounge VIP no la reconocerá.
4. Como segunda marca complementaria, elige una con precio muy alto pero presencia menor — el "secreto bien guardado" que el anfitrión revela en la suite privada.

**Sesgos a vigilar.**
- Los umbrales de Tier (Entrada < $2K, Premium $2-10K, Alta Gama $10-50K, Ultra-Lujo > $50K) son absolutos e iguales para todas las marcas. Una pieza a $8.000 es "Alta Gama" para Tudor pero "Entrada" para Patek Philippe. Valida estos umbrales con el equipo comercial antes de la presentación ejecutiva.


## 8.1. Análisis adicional de materiales

Con la recuperación conservadora anterior, `case_material` y `bracelet_material` pasan a ser más útiles para el análisis. Aun así, los gráficos siguientes usan solo registros con material informado o recuperado por referencia, para no mezclar valores desconocidos con materiales reales.


In [70]:
# ============================================================
# ANÁLISIS ADICIONAL DE MATERIALES
# ============================================================

df_materiales = df_nuevos.dropna(subset=['price', 'case_material', 'bracelet_material']).copy()

print(f"Registros con precio + material de caja + material de brazalete: {len(df_materiales):,}")
print(f"Porcentaje sobre df_nuevos limpio: {len(df_materiales) / len(df_nuevos) * 100:.1f}%")

top_case_materials = df_materiales['case_material'].value_counts().head(10).index

resumen_case = (
    df_materiales[df_materiales['case_material'].isin(top_case_materials)]
    .groupby('case_material')
    .agg(
        n_modelos=('price', 'count'),
        precio_mediano=('price', 'median'),
        precio_medio=('price', 'mean')
    )
    .reset_index()
    .sort_values('precio_mediano', ascending=False)
)

fig_mat1 = px.bar(
    resumen_case,
    x='case_material',
    y='precio_mediano',
    color='n_modelos',
    text='n_modelos',
    title='Material de caja: precio mediano y volumen disponible',
    labels={
        'case_material': 'Material de caja',
        'precio_mediano': 'Precio mediano (USD)',
        'n_modelos': 'Nº anuncios'
    },
    color_continuous_scale='Viridis'
)
fig_mat1.update_layout(template='plotly_white', xaxis_tickangle=-30, width=1100, height=600)
fig_mat1.show()

resumen_case


Registros con precio + material de caja + material de brazalete: 61,740
Porcentaje sobre df_nuevos limpio: 66.5%


,case_material,n_modelos,precio_mediano,precio_medio
8,White gold,1010,"86,007.50","99,307.16"
5,Rose gold,5207,"32,110.00","58,078.63"
9,Yellow gold,852,"21,065.50","40,187.03"
1,Carbon,245,"19,208.00","60,885.08"
4,Red gold,339,"18,979.00","22,822.59"
2,Ceramic,2583,"12,836.00","22,440.09"
7,Titanium,4905,"8,936.00","14,695.81"
3,Gold/Steel,2136,"7,938.50","9,300.68"
0,Bronze,427,"4,995.00","5,961.95"
6,Steel,43680,"4,275.00","9,134.82"


In [71]:
top_case = df_materiales['case_material'].value_counts().head(8).index
top_bracelet = df_materiales['bracelet_material'].value_counts().head(8).index

combo_materiales = (
    df_materiales[
        df_materiales['case_material'].isin(top_case) &
        df_materiales['bracelet_material'].isin(top_bracelet)
    ]
    .groupby(['case_material', 'bracelet_material'])
    .agg(
        n_modelos=('price', 'count'),
        precio_mediano=('price', 'median')
    )
    .reset_index()
)

# Evitamos combinaciones anecdóticas para que el gráfico sea accionable.
combo_materiales = combo_materiales[combo_materiales['n_modelos'] >= 30]

fig_mat2 = px.scatter(
    combo_materiales,
    x='case_material',
    y='bracelet_material',
    size='n_modelos',
    color='precio_mediano',
    hover_data=['n_modelos', 'precio_mediano'],
    title='Combinaciones caja-brazalete: volumen y precio mediano',
    labels={
        'case_material': 'Material de caja',
        'bracelet_material': 'Material de brazalete',
        'precio_mediano': 'Precio mediano (USD)',
        'n_modelos': 'Nº anuncios'
    },
    color_continuous_scale='Plasma',
    size_max=45
)
fig_mat2.update_layout(template='plotly_white', width=1100, height=650)
fig_mat2.show()

combo_materiales.sort_values(['precio_mediano', 'n_modelos'], ascending=False).head(15)


,case_material,bracelet_material,n_modelos,precio_mediano
45,White gold,Rubber,260,"101,966.00"
23,Rose gold,Rose gold,1315,"95,395.00"
43,White gold,Crocodile skin,320,"50,025.50"
44,White gold,Leather,102,"47,500.00"
51,Yellow gold,Rubber,113,"33,000.00"
24,Rose gold,Rubber,1524,"32,004.50"
20,Rose gold,Crocodile skin,1597,"26,600.00"
47,Yellow gold,Crocodile skin,173,"21,039.00"
22,Rose gold,Leather,576,"17,700.00"
4,Ceramic,Crocodile skin,157,"15,130.00"


## 9. Borrador de la sección de sesgos y gobernanza (Checklist III)

Este es un primer borrador que se ampliará y se incorporará visualmente al dashboard en el Día 3. La idea es que, en la presentación ejecutiva, esta sección se muestre como una **alerta explícita**, no como una nota técnica al pie.

### Sesgos identificados en este dataset

1. **Sesgo geográfico (el más relevante para este proyecto).**
   El dataset proviene de Chrono24, un marketplace cuyos vendedores se concentran mayoritariamente en Europa y Estados Unidos. **México y Canadá está muy subrepresentado o ausente.** Dado que la estrategia de negocio se basa en activaciones en sedes de EE.UU., Canadá *y* México, este sesgo implica que las conclusiones sobre "qué marca/tier funciona mejor" están validadas principalmente para el mercado estadounidense/europeo.
   *Impacto si se ignora:* la empresa podría replicar en los estadios de México y Canadá exactamente la misma estrategia que en EE.UU., asumiendo una demanda de marca que no está respaldada por datos locales — riesgo de mala asignación de presupuesto de marketing.

3. **Mercado secundario, no precios oficiales de venta al público (PVP).**
   Los precios reflejan reventa en páginas web, no el precio de catálogo oficial de cada marca. Para un argumento de "qué presupuesto necesita la marca para activar un stand VIP", estos precios son un proxy razonable de posicionamiento, pero **no deben presentarse como precios de tienda oficial**.


3. **Sesgo introducido por el filtro "solo nuevos" (`condition` = New/Unworn).**
   Al quedarnos solo con relojes nuevos o sin usar el dashboard podría recomendar únicamente marcas con fuerte distribución de producto nuevo a través de revendedores, penalizando a casas relojeras que comercializan su producto nuevo casi exclusivamente a través de boutiques propias (y que por tanto aparecen poco en `df_nuevos`, aunque sean líderes de mercado).

In [72]:
df_nuevos.to_csv(
    "Watches_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

In [73]:
from google.colab import files
files.download("Watches_limpio.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>